# Covalent protein–ligand complex

Apply an explicit bond constraint between KRAS G12C and CCD ligand U4U, then inspect the saved complex.

Tested with **Boltz2 NIM 1.9.0 / client 0.6.0**.
From the repository root, install `python -m pip install -e ".[notebooks]"`
and start `jupyter lab examples/notebooks` in that environment.
Use a [ready NIM server](../../docs/deployment.md); `BOLTZ2_NIM_URL`
defaults to `http://localhost:8000`. Run cells in order.

[Setup and output files](../README.md#notebooks) ·
[NIM release notes](https://docs.nvidia.com/nim/bionemo/boltz2/1.9.0/release-notes.html)

Copyright (c) 2025-2026 NVIDIA Corporation.

In [ ]:
import os
from pathlib import Path
import boltz2_client
from boltz2_client import Boltz2Client, Polymer, Ligand, PredictionRequest, BondConstraint, Atom

client = Boltz2Client(
    base_url=os.getenv("BOLTZ2_NIM_URL", "http://localhost:8000"), timeout=1200,
)
metadata = await client.get_service_metadata()
print(f"Client {boltz2_client.__version__}; connected NIM {metadata.version} (tested: 1.9.0)")
output_dir = Path("notebook_outputs") / "covalent"
output_dir.mkdir(parents=True, exist_ok=True)

## Define KRAS G12C and the ligand bond

Protein residue indices are **1-based within the supplied sequence**. Cysteine
12 atom `SG` is linked to ligand atom `C22`; ligand residue index is 1.
The [CCD U4U definition](https://files.rcsb.org/ligands/view/U4U.cif) identifies
these linkage atoms. Use CCD atom names, which can differ from SMILES atom order.

The constraint specifies the intended bond during structure prediction; it
does not estimate reaction kinetics or establish experimental covalent binding.

In [ ]:
sequence = 'MTEYKLVVVGACGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKCDLPSRTVDTKQAQDLARSYGIPFIETSAKTRQGVDDAFYTLVREIRKHKE'
assert sequence[11] == "C"
request = PredictionRequest(
    polymers=[Polymer(id="A", molecule_type="protein", sequence=sequence)],
    ligands=[Ligand(id="LIG", ccd="U4U")],
    constraints=[BondConstraint(atoms=[
        Atom(id="A", residue_index=12, atom_name="SG"),
        Atom(id="LIG", residue_index=1, atom_name="C22"),
    ])],
)
result = await client.predict(request, output_dir=output_dir)


In [ ]:
print("Saved outputs:", output_dir.resolve())
print("Confidence:", result.confidence_scores)
print("Interface TM:", result.iptm_scores)

## Inspect the binding site

The inline py3Dmol viewer highlights the ligand and cysteine side chain.
The client saves the complete response and `structure_0.cif` in the output
directory; this file can also be opened in a molecular desktop viewer.

In [ ]:
import py3Dmol

view = py3Dmol.view(width=800, height=450)
view.addModel(result.structures[0].structure, "cif")
view.setStyle({"cartoon": {"color": "lightblue", "opacity": 0.55}})
view.addStyle({"hetflag": True}, {"stick": {}})
view.addStyle({"chain": "A", "resi": 12}, {"stick": {"color": "yellow"}})
view.zoomTo({"hetflag": True})
view.zoom(0.7)
view.show()